In [1]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [2]:
# ============================================================
# 1. LOAD DATA
# ============================================================
df = sns.load_dataset("titanic")

print("=== UKURAN DATA ===")
print("Shape:", df.shape)

print("\n=== TIPE KOLOM ===")
print(df.dtypes)


=== UKURAN DATA ===
Shape: (891, 15)

=== TIPE KOLOM ===
survived          int64
pclass            int64
sex              object
age             float64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
deck           category
embark_town      object
alive            object
alone              bool
dtype: object


In [ ]:
# ============================================================
# 2. MISSING VALUE
# ============================================================
print("\n=== Missing Value (Sebelum) ===")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Alasan Penanganan:
# age → numerik → isi median (lebih robust terhadap outlier)
# embarked & embark_town → kategorikal → isi modus (nilai paling sering)
# deck → terlalu banyak kosong → isi kategori baru 'Unknown'

df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])
df["deck"] = df["deck"].astype("object").fillna("Unknown")
print("\n=== Missing Value (Sesudah) ===")
print(df.isnull().sum()[df.isnull().sum() > 0])


=== Missing Value (Sebelum) ===
age            177
embarked         2
deck           688
embark_town      2
dtype: int64

=== Missing Value (Sesudah) ===
Series([], dtype: int64)


In [4]:
# ============================================================
# 3. OUTLIER (IQR) – age & fare
# ============================================================
print("\n=== Penanganan Outlier (IQR) ===")

for col in ["age", "fare"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower, upper)
    
    print(f"{col}: {outliers} outlier ditangani")


=== Penanganan Outlier (IQR) ===
age: 66 outlier ditangani
fare: 116 outlier ditangani


In [5]:
# ============================================================
# 4. SCALING NUMERIK
# ============================================================
scaler = StandardScaler()
df[["age", "fare"]] = scaler.fit_transform(df[["age", "fare"]])

print("\n=== Statistik Setelah Scaling ===")
print(df[["age", "fare"]].describe().round(3))


=== Statistik Setelah Scaling ===
           age     fare
count  891.000  891.000
mean     0.000    0.000
std      1.001    1.001
min     -2.200   -1.175
25%     -0.583   -0.788
50%     -0.086   -0.469
75%      0.494    0.340
max      2.110    2.032


In [6]:
# ============================================================
# 5. VISUALISASI (5 Grafik)
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# 1. Distribusi Survived
df["survived"].value_counts().plot(kind="bar", ax=axes[0,0])
axes[0,0].set_title("1. Distribusi Survived")

# 2. Distribusi Age
axes[0,1].hist(df["age"], bins=20)
axes[0,1].set_title("2. Distribusi Age (Scaled)")

# 3. Distribusi Fare
axes[0,2].hist(df["fare"], bins=20)
axes[0,2].set_title("3. Distribusi Fare (Scaled)")

# 4. Survival Rate by Sex
df.groupby("sex")["survived"].mean().plot(kind="bar", ax=axes[1,0])
axes[1,0].set_title("4. Survival Rate by Sex")

# 5. Survival Rate by Class
df.groupby("class")["survived"].mean().plot(kind="bar", ax=axes[1,1])
axes[1,1].set_title("5. Survival Rate by Class")

axes[1,2].axis("off")

plt.tight_layout()
plt.savefig("eda_titanic.png")
plt.close()

print("\nVisualisasi disimpan ke eda_titanic.png")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_1996\960480491.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("class")["survived"].mean().plot(kind="bar", ax=axes[1,1])



Visualisasi disimpan ke eda_titanic.png


In [7]:
# ============================================================
# 6. 5 INSIGHT
# ============================================================

print("\n=== 5 Insight ===")

print(f"1. Tingkat survival keseluruhan adalah {df['survived'].mean():.1%}.")

print(f"2. Survival rate perempuan ({df[df['sex']=='female']['survived'].mean():.1%}) "
      f"lebih tinggi dibanding laki-laki ({df[df['sex']=='male']['survived'].mean():.1%}).")

print(f"3. Penumpang kelas First memiliki survival rate tertinggi "
      f"({df[df['class']=='First']['survived'].mean():.1%}).")

print("4. Setelah scaling, mean age dan fare mendekati 0 dan standar deviasi mendekati 1.")

print("5. Outlier pada age dan fare berhasil dikurangi menggunakan metode IQR.")

print("\n✅ Tugas selesai sesuai instruksi.")


=== 5 Insight ===
1. Tingkat survival keseluruhan adalah 38.4%.
2. Survival rate perempuan (74.2%) lebih tinggi dibanding laki-laki (18.9%).
3. Penumpang kelas First memiliki survival rate tertinggi (63.0%).
4. Setelah scaling, mean age dan fare mendekati 0 dan standar deviasi mendekati 1.
5. Outlier pada age dan fare berhasil dikurangi menggunakan metode IQR.

✅ Tugas selesai sesuai instruksi.
